<a href="https://colab.research.google.com/github/rubusarbaro/supplychain-forecast-FIME/blob/main/PIA_Prophet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

##################################################
##                                              ##
##             PRODUCTO INTEGRADOR              ##
##    Pronósticos en la Cadena de Suministro    ##
##                                              ##
##                                              ##
##  Saúl Roberto Morales Velázquez              ##
##  1856691                                     ##
##                                              ##
##################################################

#**Librerías**

In [7]:
from google.colab import userdata # Se utilizará para obtener el token del API de Banxico mediante un secreto.
import json # Permite trabajar con datos en formato JSON.
import numpy as np # Permite trabajar con opercaciones matemáticas avanzadas.
import pandas as pd # Permite trabajar con data frames.
import plotly.express as px # Librería que permite la creación de gráficas interactivas.
import requests

#**Funciones**

In [2]:
def real_plt(df: object, time_column_name: str, value_column_name: str, title: str, xaxis_title="Tiempo", yaxis_title="Ventas") :
  """
  Grafica la serie de tiempo con los datos reales.

  Args:
      df (object): DataFrame que contiene los datos a graficar.
      time_column_name (str): Nombre de la columna que contiene las fechas.
      value_column_name (str): Nombre de la columna que contiene los valores.
      title (str): Título de la gráfica.

  Returns:
      Object: Gráfica de la serie de tiempo.
  """

  fig = px.line(df, x=time_column_name, y=value_column_name, title=title)

  fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
      buttons=list([
        dict(count=1, label="1m", step="month", stepmode="backward"),
        dict(count=6, label="6m", step="month", stepmode="backward"),
        dict(count=1, label="YTD", step="year", stepmode="todate"),
        dict(count=1, label="1y", step="year", stepmode="backward"),
        dict(step="all")
      ])
    )
  )

  fig.update_layout(
    title = title,
    xaxis_title = "Tiempo",
    yaxis_title = "Tipo de cambio"
  )

  return fig.show()

In [3]:
def difference_df(df: object, data_column_name: str) :
  """
  Permite obtener la diferencia entre cada fila de datos en elo data frame.

  Args:
      df (object): DataFrame.
      data_column_name (str): Nombre de la columna que contiene los valores.

  Returns:
      Object: DataFrame con la diferencia entre cada fila.
  """

  df_diff = df
  df_diff["diff"] = np.nan

  past_number = 0
  for index, row in df.iterrows():
    if index == 0:
      df_diff.at[index, "diff"] = 0
      past_number = row[data_column_name]
    else:
      df_diff.at[index, "diff"] = (float(row[data_column_name]) / float(past_number))-1
      past_number = row[data_column_name]

  return df

In [4]:
def get_spike_dates(df: object, date_column_name: str, difference_column_name="diff") :
  avg_diff = np.mean(abs(df[difference_column_name]))
  std_diff = np.std(abs(df[difference_column_name]))

  dates = []
  for index, row in df.iterrows():
    if abs(row[difference_column_name]) > avg_diff + 3*std_diff:
      dates.append(row[date_column_name])

  return dates

#**Clases**

In [30]:
class banxico() :
  def __init__(self, token: str) :
    self.token = token
    self.serie_ID = ""
    self.url = ""
    self.response = ""
    self.HTTP_status_code = 0
    self.HTTP_status = ""
    self.json_data = ""
    self.title = ""
    self.df = pd.DataFrame()

  def get_data(self, serie_ID: str) :
    import requests # Permite realizar peticiones HTML.
    HTTP_codes = {
      "200" : "OK",
      "400" : "Bad Request",
      "401" : "Unauthorized",
      "403" : "Forbidden",
      "404" : "Not Found",
      "500" : "Internal Server Error",
    }

    self.url = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/{serie_ID}/datos?token={self.token}"

    self.response = requests.get(self.url)
    self.HTTP_status_code = self.response.status_code
    self.HTTP_status = HTTP_codes[str(self.HTTP_status_code)]

    self.json_data = self.response.json()
    self.title = self.json_data["bmx"]["series"][0]["titulo"]
    self.df = pd.DataFrame(self.json_data["bmx"]["series"][0]["datos"])

    return self.df

#**Data frame**

In [32]:
data = banxico(userdata.get('Banxico_Token'))
df = data.get_data("SF43788")
df.head()

,fecha,dato
0,02/01/1992,3.0745
1,03/01/1992,3.0723
2,06/01/1992,3.0673
3,07/01/1992,3.0650
4,08/01/1992,3.0650


In [34]:
diff_df = difference_df(df, "dato")
spike_dates = get_spike_dates(diff_df, "fecha")
spike_dates

['09/11/1993',
 '20/12/1994',
 '22/12/1994',
 '23/12/1994',
 '27/12/1994',
 '28/12/1994',
 '30/12/1994',
 '02/01/1995',
 '03/01/1995',
 '04/01/1995',
 '05/01/1995',
 '06/01/1995',
 '09/01/1995',
 '10/01/1995',
 '11/01/1995',
 '12/01/1995',
 '13/01/1995',
 '16/01/1995',
 '17/01/1995',
 '18/01/1995',
 '26/01/1995',
 '27/01/1995',
 '30/01/1995',
 '31/01/1995',
 '01/02/1995',
 '03/02/1995',
 '09/02/1995',
 '13/02/1995',
 '14/02/1995',
 '17/02/1995',
 '20/02/1995',
 '22/02/1995',
 '23/02/1995',
 '24/02/1995',
 '03/03/1995',
 '06/03/1995',
 '07/03/1995',
 '08/03/1995',
 '09/03/1995',
 '10/03/1995',
 '13/03/1995',
 '14/03/1995',
 '15/03/1995',
 '16/03/1995',
 '17/03/1995',
 '23/03/1995',
 '04/04/1995',
 '24/04/1995',
 '27/04/1995',
 '26/10/1995',
 '01/11/1995',
 '08/11/1995',
 '09/11/1995',
 '13/11/1995',
 '27/10/1997',
 '21/08/1998',
 '14/09/1998',
 '13/01/1999',
 '15/01/1999',
 '03/07/2000',
 '06/10/2008',
 '10/10/2008',
 '13/10/2008',
 '22/10/2008',
 '28/10/2008',
 '06/11/2008',
 '20/11/20

In [33]:
real_plt(df, "fecha", "dato", data.title, yaxis_title="Tipo de cambio")